In [ ]:
%pip install anthropic

In [ ]:
# Load Data

import pandas as pd

df = pd.read_csv("./exp1.csv")

In [ ]:
# Set Claude API key

import anthropic

client = anthropic.Anthropic(api_key = "")

In [ ]:
# All Currently Available Models via Claude API

models = client.models.list()

for model in models.data:
    print(model.id)

In [ ]:
# Define Prompt Template

prompt_template = """
Using the 7-point scale below, rate the acceptability of the following sentence:

1. Strongly Unacceptable
2. Unacceptable
3. Somewhat Unacceptable
4. Neutral
5. Somewhat Acceptable
6. Acceptable
7. Strongly Acceptable

Sentence: "{}"

Just provide a numerical rating (1--7) for a given sentence.
"""

In [ ]:
# Test on Individual Sentences

sentence = "John is easy to please Tom."
prompt = prompt_template.format(sentence)

response = client.messages.create(
    model = "", # Model ID
    max_tokens = 1500,
    messages = [{"role": "user", "content": prompt}]
)

print(response.content[0].text)
print(f"stop_reason: {response.stop_reason}")
print(f"input_tokens: {response.usage.input_tokens}")
print(f"output_tokens: {response.usage.output_tokens}")

In [ ]:
models = [
    "claude-opus-4-8",
    "claude-opus-4-7",
    "claude-opus-4-6",
    "claude-sonnet-4-6",
    "claude-haiku-4-5-20251001"
]

In [ ]:
# Prepare Combined Result Storage

combined_results = df.copy()

for model in models:
    combined_results[model] = ""

In [ ]:
# Run for Each Model and Populate the Respective Column

import time
from tqdm import tqdm

for model_name in models:
    print(f"Running model: {model_name}")

    for idx, row in tqdm(df.iterrows(), total = len(df), desc = f"Processing ({model_name})"):
        sentence = row["SENTENCE"]
        prompt = prompt_template.format(sentence)

        try:
            response = client.messages.create(
                model = model_name,
                max_tokens = 1500,
                messages = [{"role": "user", "content": prompt}]
            )
            rating = response.content[0].text.strip()
        except Exception as e:
            rating = f"Error: {str(e)}"

        combined_results.at[idx, model_name] = rating
        time.sleep(1.5)

In [ ]:
# Save Results

combined_results.to_csv("./exp1_claude.csv", index = False)